In [38]:
# Importing necessary libraries
from pathlib import Path
import pandas as pd

data_folder = Path("../data")
processed_folder = Path("../data/processed")
evidence_folder = Path("../outputs/evidence_tables")

scores_file = data_folder / "transparency_scores.csv"
manual_review_file = data_folder / "manual_review_notes.csv"
ai_mentions_file = evidence_folder / "ai_transparency_mentions_detailed.csv"
ranking_file = evidence_folder / "strict_transparency_score_ranking_simple.csv"

scores = pd.read_csv(scores_file, dtype={"company_id": str})
manual_review = pd.read_csv(manual_review_file, dtype={"company_id": str})
ai_mentions = pd.read_csv(ai_mentions_file, dtype={"company_id": str})
ranking = pd.read_csv(ranking_file, dtype={"company_id": str})

print("Scores loaded:", len(scores))
print("Manual review rows:", len(manual_review))
print("AI evidence extracts:", len(ai_mentions))
print("Ranking rows:", len(ranking))

ranking

Scores loaded: 20
Manual review rows: 20
AI evidence extracts: 1452
Ranking rows: 20


,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage
0,004,NatWest Group plc,210,10,61,13,92.86
1,001,Barclays plc,249,10,60,12,85.71
2,009,Admiral Group plc,103,8,39,11,78.57
3,016,Funding Circle Holdings plc,104,8,30,9,64.29
4,002,HSBC Holdings plc,102,10,39,9,64.29
5,005,Standard Chartered plc,95,7,33,9,64.29
6,003,Lloyds Banking Group plc,79,8,35,9,64.29
7,019,London Stock Exchange Group plc,119,8,34,8,57.14
8,020,Experian plc,74,7,35,7,50.00
9,006,Aviva plc,91,5,33,6,42.86


In [2]:
def show_company_evidence(company_id, keywords=None, max_rows=30):
    """
    Displays evidence extracts for one company.
    Optional: filter by selected keywords.
    """
    
    evidence = ai_mentions[
        ai_mentions["company_id"] == company_id
    ].copy()
    
    if keywords is not None:
        evidence = evidence[
            evidence["keyword"].isin(keywords)
        ]
    
    evidence = evidence[
        [
            "company_id",
            "company_name",
            "keyword",
            "page_number",
            "matched_text",
            "context"
        ]
    ].sort_values(
        by=["keyword", "page_number"]
    )
    
    return evidence.head(max_rows)

In [3]:
show_company_evidence(
    company_id="004",
    keywords=[
        "artificial intelligence",
        "AI",
        "machine learning",
        "generative AI",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_rows=40
)

,company_id,company_name,keyword,page_number,matched_text,context
457,004,NatWest Group plc,AI,11,AI,"ivity. In addition, our newly established can ..."
458,004,NatWest Group plc,AI,11,AI,"cutting-edge research, leading responsible rev..."
459,004,NatWest Group plc,AI,11,AI,"and communities across the UK. For example, gr..."
460,004,NatWest Group plc,AI,12,AI,While we have momentum across NatWest • Custom...
461,004,NatWest Group plc,AI,12,AI,the full strength of NatWest Group – using the...
462,004,NatWest Group plc,AI,22,AI,Share of AUMA in total Private Banking coders ...
463,004,NatWest Group plc,AI,24,AI,to keep customers safe UK inflation rose to 3....
464,004,NatWest Group plc,AI,24,AI,"a of Excellence, we continued to develop Unemp..."
465,004,NatWest Group plc,AI,24,AI,"our defensive capabilities against the slowed,..."
466,004,NatWest Group plc,AI,25,AI,"transformation, driven by accelerated advancem..."


In [4]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

natwest_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "004"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

natwest_keyword_counts

,keyword,mention_count,unique_pages
0,AI,144,42
2,artificial intelligence,27,14
4,automation,15,12
7,generative AI,10,9
9,responsible AI,4,3
5,bias,4,4
8,machine learning,2,2
6,fairness,2,2
1,AI ethics,1,1
3,automated decision-making,1,1


In [5]:
def print_company_evidence(company_id, keywords, max_items=25):
    evidence = ai_mentions[
        (ai_mentions["company_id"] == company_id) &
        (ai_mentions["keyword"].isin(keywords))
    ].copy()
    
    evidence = evidence.sort_values(
        by=["keyword", "page_number"]
    )
    
    for _, row in evidence.head(max_items).iterrows():
        print("=" * 100)
        print(f"Company: {row['company_name']}")
        print(f"Keyword: {row['keyword']}")
        print(f"Page: {row['page_number']}")
        print("-" * 100)
        print(row["context"])
        print()

In [6]:
print_company_evidence(
    company_id="004",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=30
)

Company: NatWest Group plc
Keyword: AI ethics
Page: 49
----------------------------------------------------------------------------------------------------
ed on appropriate guardrails to ensure thebest practice with an expectation that suppliers (now 21, as at 31 December 2025). As a result, the University’s Data and AI Ethics masters safe and secure deployment of AI.(1)conform to the most recent version of the we have seen around a 20-percentage-point course, with more cohorts planned for 2026.

Company: NatWest Group plc
Keyword: artificial intelligence
Page: 59
----------------------------------------------------------------------------------------------------
for example, privacy training to our data customers and suppliers. and artificial intelligence colleagues. Regular risk assessments: Risk assessments During 2025, there were a small number are condu

Company: NatWest Group plc
Keyword: artificial intelligence
Page: 80
----------------------------------------------------------

In [7]:
# Manual validation outcome for NatWest

manual_review.loc[
    manual_review["company_id"] == "004",
    "manual_adjusted_score"
] = 13

manual_review.loc[
    manual_review["company_id"] == "004",
    "manual_adjusted_percentage"
] = 92.86

manual_review.loc[
    manual_review["company_id"] == "004",
    "reason_for_adjustment"
] = (
    "No adjustment made. NatWest shows repeated AI-related disclosures, "
    "including artificial intelligence, generative AI, AI ethics, risk scenarios, "
    "automated decision-making and risk/control language. Some extracted contexts "
    "are noisy, but the overall evidence supports a high provisional transparency score."
)

manual_review.loc[
    manual_review["company_id"] == "004",
    "important_pages"
] = "49, 80, 190, 413-420"

manual_review.loc[
    manual_review["company_id"] == "004",
    "important_extracts"
] = (
    "References to AI ethics, responsible guardrails, generative AI, "
    "AI-related risk scenarios, artificial intelligence usage and automated decision-making."
)

manual_review.loc[
    manual_review["company_id"] == "004",
    "review_status"
] = "Reviewed - provisional score retained"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "004"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
0,004,NatWest Group plc,210,10,61,13,92.86,13.0,92.86,"No adjustment made. NatWest shows repeated AI-related disclosures, including artificial intelligence, generative AI, AI ethics, risk scenarios, automated decision-making and risk/control language. Some extracted contexts are noisy, but the overall evidence supports a high provisional transparency score.","49, 80, 190, 413-420","References to AI ethics, responsible guardrails, generative AI, AI-related risk scenarios, artificial intelligence usage and automated decision-making.",Reviewed - provisional score retained


In [8]:
# Fixing manual review column data types

text_columns = [
    "reason_for_adjustment",
    "important_pages",
    "important_extracts",
    "review_status"
]

numeric_columns = [
    "manual_adjusted_score",
    "manual_adjusted_percentage"
]

# Make sure text columns can accept written notes
for column in text_columns:
    if column not in manual_review.columns:
        manual_review[column] = ""
    manual_review[column] = manual_review[column].astype("object")

# Make sure numeric columns can accept score values
for column in numeric_columns:
    if column not in manual_review.columns:
        manual_review[column] = pd.NA

# Now record manual validation outcome for NatWest again

manual_review.loc[
    manual_review["company_id"] == "004",
    "manual_adjusted_score"
] = 13

manual_review.loc[
    manual_review["company_id"] == "004",
    "manual_adjusted_percentage"
] = 92.86

manual_review.loc[
    manual_review["company_id"] == "004",
    "reason_for_adjustment"
] = (
    "No adjustment made. NatWest shows repeated AI-related disclosures, "
    "including artificial intelligence, generative AI, AI ethics, risk scenarios, "
    "automated decision-making and risk/control language. Some extracted contexts "
    "are noisy, but the overall evidence supports a high provisional transparency score."
)

manual_review.loc[
    manual_review["company_id"] == "004",
    "important_pages"
] = "49, 80, 190, 413-420"

manual_review.loc[
    manual_review["company_id"] == "004",
    "important_extracts"
] = (
    "References to AI ethics, responsible guardrails, generative AI, "
    "AI-related risk scenarios, artificial intelligence usage and automated decision-making."
)

manual_review.loc[
    manual_review["company_id"] == "004",
    "review_status"
] = "Reviewed - provisional score retained"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "004"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
0,004,NatWest Group plc,210,10,61,13,92.86,13.0,92.86,"No adjustment made. NatWest shows repeated AI-related disclosures, including artificial intelligence, generative AI, AI ethics, risk scenarios, automated decision-making and risk/control language. Some extracted contexts are noisy, but the overall evidence supports a high provisional transparency score.","49, 80, 190, 413-420","References to AI ethics, responsible guardrails, generative AI, AI-related risk scenarios, artificial intelligence usage and automated decision-making.",Reviewed - provisional score retained


In [9]:
# Barclays keyword count review

barclays_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "001"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

barclays_keyword_counts

,keyword,mention_count,unique_pages
0,AI,195,47
1,artificial intelligence,20,14
3,bias,8,6
2,automation,7,7
5,fairness,7,7
8,machine learning,7,6
6,generative AI,2,2
4,ethical AI,1,1
7,human oversight,1,1
9,responsible AI,1,1


In [10]:
print_company_evidence(
    company_id="001",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: Barclays plc
Keyword: artificial intelligence
Page: 42
----------------------------------------------------------------------------------------------------
class customer experience; and delivering best You can find out more about how the Directors have had regard to the matters set out in Section embracing artificial intelligence (AI) and other new in classservicecustomer 172(1) when discharging their duties, and the technologies in

Company: Barclays plc
Keyword: artificial intelligence
Page: 62
----------------------------------------------------------------------------------------------------
ds (including business or operations; the use of newor current facts. Forward-looking statements emissions accounting methodologies); changes technology, including artificial intelligence; thesometimes use words such as ‘may’, ‘will’, ‘seek’, in tax laws and practice; the outcome of current Group’s ability to access funding; and the success‘

Company: Barclays plc
Keyword: artificial 

Some fairness related matches were not AI-specific and were therefore treated cautiously during manual validation.

In [11]:
# Manual validation outcome for Barclays

manual_review.loc[
    manual_review["company_id"] == "001",
    "manual_adjusted_score"
] = 12

manual_review.loc[
    manual_review["company_id"] == "001",
    "manual_adjusted_percentage"
] = 85.71

manual_review.loc[
    manual_review["company_id"] == "001",
    "reason_for_adjustment"
] = (
    "No adjustment made. Barclays shows meaningful AI transparency evidence, "
    "including Ethical AI Principles, AI Policy references, model strategy and oversight language, "
    "AI bias and hallucination risks, generative AI, agentic AI, machine learning and AI-related risk controls. "
    "However, some fairness matches were not AI-specific and relate instead to remuneration, employees or fair lending, "
    "so fairness evidence was treated cautiously."
)

manual_review.loc[
    manual_review["company_id"] == "001",
    "important_pages"
] = "183, 233, 238-241, 253, 342, 355, 382"

manual_review.loc[
    manual_review["company_id"] == "001",
    "important_extracts"
] = (
    "References to Ethical AI Principles, AI Policy, AI bias, hallucinations, "
    "generative AI, agentic AI, machine learning, AI-related risk controls and AI/ML initiatives."
)

manual_review.loc[
    manual_review["company_id"] == "001",
    "review_status"
] = "Reviewed - provisional score retained"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "001"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
1,001,Barclays plc,249,10,60,12,85.71,12.0,85.71,"No adjustment made. Barclays shows meaningful AI transparency evidence, including Ethical AI Principles, AI Policy references, model strategy and oversight language, AI bias and hallucination risks, generative AI, agentic AI, machine learning and AI-related risk controls. However, some fairness matches were not AI-specific and relate instead to remuneration, employees or fair lending, so fairness evidence was treated cautiously.","183, 233, 238-241, 253, 342, 355, 382","References to Ethical AI Principles, AI Policy, AI bias, hallucinations, generative AI, agentic AI, machine learning, AI-related risk controls and AI/ML initiatives.",Reviewed - provisional score retained


In [12]:
# Admiral keyword count review

admiral_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "009"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

admiral_keyword_counts

,keyword,mention_count,unique_pages
0,AI,71,34
1,artificial intelligence,8,6
7,responsible AI,7,4
4,fairness,5,5
6,machine learning,5,4
2,automation,3,3
5,generative AI,3,3
3,bias,1,1


In [13]:
print_company_evidence(
    company_id="009",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: Admiral Group plc
Keyword: artificial intelligence
Page: 10
----------------------------------------------------------------------------------------------------
ese shifts by leveraging our strengths across customer centricity, underwriting excellence, agility, and innovation. Generative and agentic UK direct insurance market artificial intelligence Generative and agentic artificial intelligence (‘AI’) is Over the past ten years, the UK direct insurance market fundamentally reshaping consumer behaviours across has demonstrated consistent growth,

Company: Admiral Group plc
Keyword: artificial intelligence
Page: 10
----------------------------------------------------------------------------------------------------
stomer centricity, underwriting excellence, agility, and innovation. Generative and agentic UK direct insurance market artificial intelligence Generative and agentic artificial intelligence (‘AI’) is Over the past ten years, the UK direct insurance market fundamentall

In [14]:
# Ensuring manual review columns can hold text values
text_columns = [
    "reason_for_adjustment",
    "important_pages",
    "important_extracts",
    "review_status"
]

for col in text_columns:
    if col not in manual_review.columns:
        manual_review[col] = ""
    manual_review[col] = manual_review[col].astype("object")

numeric_columns = [
    "manual_adjusted_score",
    "manual_adjusted_percentage"
]

for col in numeric_columns:
    if col not in manual_review.columns:
        manual_review[col] = pd.NA

# Make sure company IDs are treated consistently
manual_review["company_id"] = manual_review["company_id"].astype(str).str.zfill(3)

# Record manual validation outcome for Admiral Group
admiral_mask = manual_review["company_id"] == "009"

manual_review.loc[
    admiral_mask,
    "manual_adjusted_score"
] = 11

manual_review.loc[
    admiral_mask,
    "manual_adjusted_percentage"
] = 78.57

manual_review.loc[
    admiral_mask,
    "reason_for_adjustment"
] = (
    "No adjustment made. Admiral Group shows strong AI transparency evidence, including responsible AI, "
    "AI governance standards, artificial intelligence, generative AI, agentic AI, machine learning, predictive AI, "
    "and AI use across pricing, claims, underwriting, customer retention, data quality and internal operations. "
    "The provisional score is retained because the evidence supports AI use, business application, risk, governance, "
    "ethics/fairness and specificity. However, explicit human oversight or human-in-the-loop controls were not clearly disclosed."
)

manual_review.loc[
    admiral_mask,
    "important_pages"
] = "10, 19, 21, 22, 36, 44, 49, 66, 99, 130, 157"

manual_review.loc[
    admiral_mask,
    "important_extracts"
] = (
    "Responsible AI deployment, AI governance standards, GenAI Centre of Excellence, generative and agentic AI, "
    "machine learning in pricing, predictive AI models, claims, risk selection, customer retention, data quality, "
    "fairness, openness, transparency and ethics."
)

manual_review.loc[
    admiral_mask,
    "review_status"
] = "Reviewed - provisional score retained"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "009"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
2,009,Admiral Group plc,103,8,39,11,78.57,11.0,78.57,"No adjustment made. Admiral Group shows strong AI transparency evidence, including responsible AI, AI governance standards, artificial intelligence, generative AI, agentic AI, machine learning, predictive AI, and AI use across pricing, claims, underwriting, customer retention, data quality and internal operations. The provisional score is retained because the evidence supports AI use, business application, risk, governance, ethics/fairness and specificity. However, explicit human oversight or human-in-the-loop controls were not clearly disclosed.","10, 19, 21, 22, 36, 44, 49, 66, 99, 130, 157","Responsible AI deployment, AI governance standards, GenAI Centre of Excellence, generative and agentic AI, machine learning in pricing, predictive AI models, claims, risk selection, customer retention, data quality, fairness, openness, transparency and ethics.",Reviewed - provisional score retained


In [15]:
# Funding Circle keyword count review

funding_circle_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "016"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

funding_circle_keyword_counts

,keyword,mention_count,unique_pages
0,AI,91,27
2,automation,3,3
3,bias,3,2
1,artificial intelligence,2,2
7,machine learning,2,2
4,fairness,1,1
5,generative AI,1,1
6,human-in-the-loop,1,1


In [16]:
print_company_evidence(
    company_id="016",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: Funding Circle Holdings plc
Keyword: artificial intelligence
Page: 59
----------------------------------------------------------------------------------------------------
ats: the dual nature of AI environment. By maintaining a disciplined and cybersecurity approach to risk while responsibly risk landscape of 2026.” Artificial intelligence is reshaping financial embracing innovation, we will support services. For us, AI is a powerful lever UK SMEs through the cycle and create

Company: Funding Circle Holdings plc
Keyword: artificial intelligence
Page: 63
----------------------------------------------------------------------------------------------------
ness. Risk appetite We will make efficient use of our available resources to build a sustainable, diversified and profitable business that can successfully adapt to environment and technological changes (in particular artificial intelligence (‘‘AI’’)), and respond adequately to competition pressures. Risk(s) and potential impac

In [17]:
# Ensuring manual review columns can hold text values
text_columns = [
    "reason_for_adjustment",
    "important_pages",
    "important_extracts",
    "review_status"
]

for col in text_columns:
    if col not in manual_review.columns:
        manual_review[col] = ""
    manual_review[col] = manual_review[col].astype("object")

numeric_columns = [
    "manual_adjusted_score",
    "manual_adjusted_percentage"
]

for col in numeric_columns:
    if col not in manual_review.columns:
        manual_review[col] = pd.NA

# Make sure company IDs are treated consistently
manual_review["company_id"] = manual_review["company_id"].astype(str).str.zfill(3)

# Record manual validation outcome for Funding Circle
funding_circle_mask = manual_review["company_id"] == "016"

manual_review.loc[
    funding_circle_mask,
    "manual_adjusted_score"
] = 9

manual_review.loc[
    funding_circle_mask,
    "manual_adjusted_percentage"
] = 64.29

manual_review.loc[
    funding_circle_mask,
    "reason_for_adjustment"
] = (
    "No adjustment made. Funding Circle shows relevant AI transparency evidence,machine learning, an automated decision engine, AI/ML models used in credit risk assessment, and AI-related risk language. "
    "The report also refers to risks from flawed AI model outputs, process automation failures, AI-enabled fraud and inadequate "
    "human-in-the-loop oversight. However, the evidence remains weaker on detailed AI governance, AI-specific fairness and "
    "clear operational human review controls."
)

manual_review.loc[
    funding_circle_mask,
    "important_pages"
] = "8, 24, 31, 59, 63, 66, 68, 125"

manual_review.loc[
    funding_circle_mask,
    "important_extracts"
] = (
    "Generative AI applications, machine learning, automated decision engine, complex underwriting, AI/ML models in credit risk "
    "assessment, AI-related operational risk, flawed model outputs, process automation failures, AI-enabled fraud, human-in-the-loop "
    "oversight, fairness and bias references."
)

manual_review.loc[
    funding_circle_mask,
    "review_status"
] = "Reviewed - provisional score retained"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "016"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
3,016,Funding Circle Holdings plc,104,8,30,9,64.29,9.0,64.29,"No adjustment made. Funding Circle shows relevant AI transparency evidence,machine learning, an automated decision engine, AI/ML models used in credit risk assessment, and AI-related risk language. The report also refers to risks from flawed AI model outputs, process automation failures, AI-enabled fraud and inadequate human-in-the-loop oversight. However, the evidence remains weaker on detailed AI governance, AI-specific fairness and clear operational human review controls.","8, 24, 31, 59, 63, 66, 68, 125","Generative AI applications, machine learning, automated decision engine, complex underwriting, AI/ML models in credit risk assessment, AI-related operational risk, flawed model outputs, process automation failures, AI-enabled fraud, human-in-the-loop oversight, fairness and bias references.",Reviewed - provisional score retained


In [18]:
# HSBC keyword count review

hsbc_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "002"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

hsbc_keyword_counts

,keyword,mention_count,unique_pages
0,AI,75,29
2,artificial intelligence,5,5
8,machine learning,4,3
3,automation,4,4
4,bias,4,4
6,generative AI,4,4
5,fairness,3,3
1,AI ethics,1,1
7,large language model,1,1
9,responsible AI,1,1


In [19]:
print_company_evidence(
    company_id="002",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: HSBC Holdings plc
Keyword: AI ethics
Page: 58
----------------------------------------------------------------------------------------------------
et future challenges. and their level of AI involvement. It provides comprehensive training on AI literacy, Establishing our leadership framework responsible AI, and AI ethics, with To support our refreshed strategy and participants earning badges to recognise their ambition, a cross-section of business leaders achievements. In 2025, we piloted the AI developed and launched a

Company: HSBC Holdings plc
Keyword: artificial intelligence
Page: 13
----------------------------------------------------------------------------------------------------
source of funding for us, and forms the Hongkong and Shanghai Banking Corporation foundation of our financial stability. Limited, HSBC UK and HSBC Bank plc. Improving operational excellence through artificial intelligence In 2025, we accelerated the adoption of tool, HSBC Productivity Suite, w

In [20]:
# Ensuring manual review columns can hold text values
text_columns = [
    "reason_for_adjustment",
    "important_pages",
    "important_extracts",
    "review_status"
]

for col in text_columns:
    if col not in manual_review.columns:
        manual_review[col] = ""
    manual_review[col] = manual_review[col].astype("object")

numeric_columns = [
    "manual_adjusted_score",
    "manual_adjusted_percentage"
]

for col in numeric_columns:
    if col not in manual_review.columns:
        manual_review[col] = pd.NA

# Make sure company IDs are treated consistently
manual_review["company_id"] = manual_review["company_id"].astype(str).str.zfill(3)

# Record manual validation outcome for HSBC
hsbc_mask = manual_review["company_id"] == "002"

manual_review.loc[
    hsbc_mask,
    "manual_adjusted_score"
] = 12

manual_review.loc[
    hsbc_mask,
    "manual_adjusted_percentage"
] = 85.71

manual_review.loc[
    hsbc_mask,
    "reason_for_adjustment"
] = (
    "Score increased from provisional automated score. HSBC shows strong evidence of AI transparency through operational GenAI "
    "adoption, HSBC Productivity Suite, machine learning, AI-supported operational excellence, model risk governance, Board and "
    "committee oversight, responsible AI, AI ethics and AI literacy training. However, direct evidence of human-in-the-loop review "
    "for AI decisions is limited, and some bias/fairness matches relate to general conduct, pay or accounting fairness rather than "
    "AI-specific fairness."
)

manual_review.loc[
    hsbc_mask,
    "important_pages"
] = "13, 58, 61, 105, 106, 183, 217, 368"

manual_review.loc[
    hsbc_mask,
    "important_extracts"
] = (
    "HSBC Productivity Suite, Generative AI adoption, document analysis and translation, machine learning, GenAI techniques, "
    "model validation, model risk, Group Model Risk, Board oversight, Group Technology & Operations Committee review, responsible AI, "
    "AI ethics, AI literacy training, ethical use of data, AI and machine learning risk monitoring."
)

manual_review.loc[
    hsbc_mask,
    "review_status"
] = "Reviewed - score increased"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "002"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
4,002,HSBC Holdings plc,102,10,39,9,64.29,12.0,85.71,"Score increased from provisional automated score. HSBC shows strong evidence of AI transparency through operational GenAI adoption, HSBC Productivity Suite, machine learning, AI-supported operational excellence, model risk governance, Board and committee oversight, responsible AI, AI ethics and AI literacy training. However, direct evidence of human-in-the-loop review for AI decisions is limited, and some bias/fairness matches relate to general conduct, pay or accounting fairness rather than AI-specific fairness.","13, 58, 61, 105, 106, 183, 217, 368","HSBC Productivity Suite, Generative AI adoption, document analysis and translation, machine learning, GenAI techniques, model validation, model risk, Group Model Risk, Board oversight, Group Technology & Operations Committee review, responsible AI, AI ethics, AI literacy training, ethical use of data, AI and machine learning risk monitoring.",Reviewed - score increased


In [21]:
# Standard Chartered keyword count review

standard_chartered_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "005"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

standard_chartered_keyword_counts

,keyword,mention_count,unique_pages
0,AI,80,26
1,artificial intelligence,3,3
2,automation,3,2
4,fairness,3,3
6,responsible AI,3,1
3,bias,2,2
5,generative AI,1,1


In [22]:
print_company_evidence(
    company_id="005",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: Standard Chartered plc
Keyword: artificial intelligence
Page: 148
----------------------------------------------------------------------------------------------------
ty, sustainability challenges and Group’s strategy, the activities of the Board leadership succession. committees and a keynote presentation on artificial intelligence and cyber governance. AGM The AGM, held on 8 May in 2025, was the Board’s Provided an opportunity for retail shareholders key opportunity for engagement with retail to engag

Company: Standard Chartered plc
Keyword: artificial intelligence
Page: 195
----------------------------------------------------------------------------------------------------
the traditional bank by addressing the Ventures and the rest digital banking and lifestyle needs of clients. of the Bank • Bill championed our Group approach to artificial intelligence (AI) with 15 themes identified for execution, including ‘MyWealth Advisor’ in Singapore and Hong Kong. • Continue to dev

In [23]:
# Ensuring manual review columns can hold text values
text_columns = [
    "reason_for_adjustment",
    "important_pages",
    "important_extracts",
    "review_status"
]

for col in text_columns:
    if col not in manual_review.columns:
        manual_review[col] = ""
    manual_review[col] = manual_review[col].astype("object")

numeric_columns = [
    "manual_adjusted_score",
    "manual_adjusted_percentage"
]

for col in numeric_columns:
    if col not in manual_review.columns:
        manual_review[col] = pd.NA

manual_review["company_id"] = manual_review["company_id"].astype(str).str.zfill(3)

# Record manual validation outcome for Standard Chartered
standard_chartered_mask = manual_review["company_id"] == "005"

manual_review.loc[
    standard_chartered_mask,
    "manual_adjusted_score"
] = 12

manual_review.loc[
    standard_chartered_mask,
    "manual_adjusted_percentage"
] = 85.71

manual_review.loc[
    standard_chartered_mask,
    "reason_for_adjustment"
] = (
    "Score increased from provisional automated score. Standard Chartered shows AI-specific transparency evidence through "
    "responsible AI governance, annual Code recommitment, Audit Committee reporting on supplier and data risk including "
    "responsible AI, AI-related business execution themes, MyWealth Advisor, GenAI/digitalisation references, and AI-related "
    "risk disclosure. However, direct evidence of human-in-the-loop review is limited, and some fairness/bias matches relate "
    "to general remuneration, inclusion or accounting fairness rather than AI-specific fairness."
)

manual_review.loc[
    standard_chartered_mask,
    "important_pages"
] = "16, 148, 187, 195, 199, 218, 288, 320"

manual_review.loc[
    standard_chartered_mask,
    "important_extracts"
] = (
    "Responsible AI governance established; employees and directors recommit to the Code annually; Audit Committee receives "
    "reports on supplier and data risk including responsible AI; AI approach with 15 themes identified for execution; "
    "MyWealth Advisor; digitalisation including GenAI adoption; adoption and use of artificial intelligence as a risk; "
    "MAS FEAT-style fairness, ethics, accountability and transparency references."
)

manual_review.loc[
    standard_chartered_mask,
    "review_status"
] = "Reviewed - score increased"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "005"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
5,005,Standard Chartered plc,95,7,33,9,64.29,12.0,85.71,"Score increased from provisional automated score. Standard Chartered shows AI-specific transparency evidence through responsible AI governance, annual Code recommitment, Audit Committee reporting on supplier and data risk including responsible AI, AI-related business execution themes, MyWealth Advisor, GenAI/digitalisation references, and AI-related risk disclosure. However, direct evidence of human-in-the-loop review is limited, and some fairness/bias matches relate to general remuneration, inclusion or accounting fairness rather than AI-specific fairness.","16, 148, 187, 195, 199, 218, 288, 320","Responsible AI governance established; employees and directors recommit to the Code annually; Audit Committee receives reports on supplier and data risk including responsible AI; AI approach with 15 themes identified for execution; MyWealth Advisor; digitalisation including GenAI adoption; adoption and use of artificial intelligence as a risk; MAS FEAT-style fairness, ethics, accountability and transparency references.",Reviewed - score increased


In [24]:
# Lloyds keyword count review

lloyds_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "003"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

lloyds_keyword_counts

,keyword,mention_count,unique_pages
0,AI,58,28
2,artificial intelligence,9,8
6,generative AI,5,5
3,automation,3,3
1,AI ethics,1,1
4,bias,1,1
5,ethical AI,1,1
7,machine learning,1,1


In [25]:
print_company_evidence(
    company_id="003",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: Lloyds Banking Group plc
Keyword: AI ethics
Page: 83
----------------------------------------------------------------------------------------------------
agriculture • Wider sustainability topics included: the Group’s treatment of vulnerable customers, generative AI deep dive which outlined the Group’s AI ethics principles and how use cases are overseen via the Data and AI Ethics Committee, economic crime deep dive and key drivers of people risk and mitigating action Audit

Company: Lloyds Banking Group plc
Keyword: artificial intelligence
Page: 14
----------------------------------------------------------------------------------------------------
verview Competition remains intense with high street banks and Rapidly evolving technology landscape, accelerated building societies maintaining their focus on share growth, by developments in artificial intelligence and digital and building scale by consolidating smaller players. Alongside, transformation. These shifts are enabling 

In [26]:
# Manual validation outcome for Lloyds Banking Group plc

lloyds_mask = manual_review["company_id"] == "003"

manual_review.loc[
    lloyds_mask,
    "manual_adjusted_score"
] = 12

manual_review.loc[
    lloyds_mask,
    "manual_adjusted_percentage"
] = 85.71

manual_review.loc[
    lloyds_mask,
    "reason_for_adjustment"
] = (
    "Score increased from provisional automated score. Lloyds shows stronger AI-specific transparency evidence than the "
    "automated score captured, including Board-level discussion of generative AI and generative AI initiatives. Business-use, governance and ethics/fairness scores were therefore increased. "
    "However, direct evidence of operational human-in-the-loop review or manual review of AI decisions remains limited."
)

manual_review.loc[
    lloyds_mask,
    "important_pages"
] = "26, 29, 31, 43, 77, 80, 83, 85, 94, 211"

manual_review.loc[
    lloyds_mask,
    "important_extracts"
] = (
    "Board discussed the pace and impact of generative AI; generative AI deep dive outlined AI ethics principles; use cases "
    "overseen via the Data and AI Ethics Committee; data ethics framework and Ethical AI implemented; model risk management "
    "supports safe and strategic development of AI and machine learning applications; digital and artificial intelligence tools "
    "support customers; Board received updates on data, digital assets, technology and AI; oversight and challenge over model "
    "and data risk."
)

manual_review.loc[
    lloyds_mask,
    "review_status"
] = "Reviewed - score increased"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "003"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
6,003,Lloyds Banking Group plc,79,8,35,9,64.29,12.0,85.71,"Score increased from provisional automated score. Lloyds shows stronger AI-specific transparency evidence than the automated score captured, including Board-level discussion of generative AI and generative AI initiatives. Business-use, governance and ethics/fairness scores were therefore increased. However, direct evidence of operational human-in-the-loop review or manual review of AI decisions remains limited.","26, 29, 31, 43, 77, 80, 83, 85, 94, 211","Board discussed the pace and impact of generative AI; generative AI deep dive outlined AI ethics principles; use cases overseen via the Data and AI Ethics Committee; data ethics framework and Ethical AI implemented; model risk management supports safe and strategic development of AI and machine learning applications; digital and artificial intelligence tools support customers; Board received updates on data, digital assets, technology and AI; oversight and challenge over model and data risk.",Reviewed - score increased


In [27]:
# LSEG keyword count review

lseg_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "019"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

lseg_keyword_counts

,keyword,mention_count,unique_pages
0,AI,96,30
2,artificial intelligence,7,6
4,bias,7,5
3,automation,4,3
1,LLM,2,2
5,generative AI,1,1
6,large language model,1,1
7,machine learning,1,1


In [28]:
print_company_evidence(
    company_id="019",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "large language model",
        "LLM",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: London Stock Exchange Group plc
Keyword: LLM
Page: 13
----------------------------------------------------------------------------------------------------
n business, so we Interoperability is in our DNA. When other in financial services through professionals work, with can innovate faster and exchange groups focused on vertical our open, LLM-agnostic AI-enabled products that serve our customers better. integration of trading and clearing, we partnership approach. bring speed, simplicity champio

Company: London Stock Exchange Group plc
Keyword: LLM
Page: 199
----------------------------------------------------------------------------------------------------
insurance products. FTSE Russell FTSE International Limited and its subsidiaries, LLM the Group subsidiary that is a leading global Large language model. provider of index and analytics solutions.

Company: London Stock Exchange Group plc
Keyword: artificial intelligence
Page: 8
--------------------------------------------

In [29]:
# Manual validation outcome for London Stock Exchange Group plc

lseg_mask = manual_review["company_id"] == "019"

manual_review.loc[
    lseg_mask,
    "manual_adjusted_score"
] = 11

manual_review.loc[
    lseg_mask,
    "manual_adjusted_percentage"
] = 78.57

manual_review.loc[
    lseg_mask,
    "reason_for_adjustment"
] = (
    "Score increased from provisional automated score. LSEG shows stronger AI-specific transparency evidence than the automated score captured. "
    "Governance and risk scores were therefore increased. However, fairness and bias references were mostly general HR, pay, "
    "audit judgement or legal references. Human oversight evidence remains partial because the report refers to independent model review. "
)

manual_review.loc[
    lseg_mask,
    "important_pages"
] = "8, 13, 53, 55, 56, 67, 108, 199"

manual_review.loc[
    lseg_mask,
    "important_extracts"
] = (
    "Artificial intelligence is transforming financial markets; LSEG refers to AI-enabled products that serve customers better; "
    "cloud-based generative AI capabilities are considered in development and deployment; language models and LLM-agnostic "
    "AI-enabled products are referenced; AI is linked to risk frameworks, model risk, generative AI risk and agentic models; "
    "the Model Risk Committee and regular independent review provide evidence of governance and oversight."
)

manual_review.loc[
    lseg_mask,
    "review_status"
] = "Reviewed - score increased"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "019"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
7,019,London Stock Exchange Group plc,119,8,34,8,57.14,11.0,78.57,"Score increased from provisional automated score. LSEG shows stronger AI-specific transparency evidence than the automated score captured. Governance and risk scores were therefore increased. However, fairness and bias references were mostly general HR, pay, audit judgement or legal references. Human oversight evidence remains partial because the report refers to independent model review.","8, 13, 53, 55, 56, 67, 108, 199","Artificial intelligence is transforming financial markets; LSEG refers to AI-enabled products that serve customers better; cloud-based generative AI capabilities are considered in development and deployment; language models and LLM-agnostic AI-enabled products are referenced; AI is linked to risk frameworks, model risk, generative AI risk and agentic models; the Model Risk Committee and regular independent review provide evidence of governance and oversight.",Reviewed - score increased


In [30]:
# Experian keyword count review

experian_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "020"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

experian_keyword_counts

,keyword,mention_count,unique_pages
0,AI,44,23
1,artificial intelligence,8,7
2,automation,7,7
5,generative AI,6,5
4,fairness,5,5
3,bias,2,2
6,machine learning,2,2


In [31]:
print_company_evidence(
    company_id="020",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "large language model",
        "LLM",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: Experian plc
Keyword: artificial intelligence
Page: 6
----------------------------------------------------------------------------------------------------
rating of 4.2/5, Our client Net Promotor Score (NPS) has up from 4.1 four years ago increased for seven years running since FY18 Our Generative Artificial Intelligence >200m (GenAI)-enabled tool, Experian Assistant, Our free consumer member won the 2025 BIG Innovation Award in the

Company: Experian plc
Keyword: artificial intelligence
Page: 13
----------------------------------------------------------------------------------------------------
data – we combine it with powerful software, of Group revenue* analytics, and Artificial Intelligence (AI) to create solutions.What we do We help millions of people take control of their finances. We provide credit We integ

Company: Experian plc
Keyword: artificial intelligence
Page: 83
---------------------------------------------------------------------------------------------------

In [32]:
# Manual validation outcome for Experian plc

experian_mask = manual_review["company_id"] == "020"

manual_review.loc[
    experian_mask,
    "manual_adjusted_score"
] = 11

manual_review.loc[
    experian_mask,
    "manual_adjusted_percentage"
] = 78.57

manual_review.loc[
    experian_mask,
    "reason_for_adjustment"
] = (
    "Score increased from provisional automated score. Experian shows stronger AI-specific transparency evidence than the "
    "automated score captured, including machine learning in credit scoring, and GenAI-related risks. Governance, risk, business use and specificity were therefore "
    "increased. However, most bias references were related to audit judgement, fraud risk or workforce "
    "inclusion. Human oversight evidence remains limited because the report does not clearly describe manual intervention in AI-driven decisions."
)

manual_review.loc[
    experian_mask,
    "important_pages"
] = "20, 31, 58, 60, 62, 63, 109, 121"

manual_review.loc[
    experian_mask,
    "important_extracts"
] = (
    "Experian refers to Generative AI technologies including financial assistant EVA, which helps members understand their credit data; "
    "Experian Lift Premium applies machine learning and other advanced techniques to increase credit access for credit-invisible consumers; "
    "the company states that it champions responsible use of AI, including machine learning and GenAI, to enhance productivity, drive innovation "
    "and improve customer solutions; the report references a clear approach to GenAI, including reliability, security and performance; "
    "Board updates include privacy and artificial intelligence, consumer credit and regulatory trends."
)

manual_review.loc[
    experian_mask,
    "review_status"
] = "Reviewed - score increased"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "020"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
8,020,Experian plc,74,7,35,7,50.0,11.0,78.57,"Score increased from provisional automated score. Experian shows stronger AI-specific transparency evidence than the automated score captured, including machine learning in credit scoring, and GenAI-related risks. Governance, risk, business use and specificity were therefore increased. However, most bias references were related to audit judgement, fraud risk or workforce inclusion. Human oversight evidence remains limited because the report does not clearly describe manual intervention in AI-driven decisions.","20, 31, 58, 60, 62, 63, 109, 121","Experian refers to Generative AI technologies including financial assistant EVA, which helps members understand their credit data; Experian Lift Premium applies machine learning and other advanced techniques to increase credit access for credit-invisible consumers; the company states that it champions responsible use of AI, including machine learning and GenAI, to enhance productivity, drive innovation and improve customer solutions; the report references a clear approach to GenAI, including reliability, security and performance; Board updates include privacy and artificial intelligence, consumer credit and regulatory trends.",Reviewed - score increased


In [33]:
# Aviva keyword count review

aviva_keyword_counts = (
    ai_mentions[ai_mentions["company_id"] == "006"]
    .groupby("keyword")
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(by="mention_count", ascending=False)
)

aviva_keyword_counts

,keyword,mention_count,unique_pages
0,AI,65,26
1,artificial intelligence,17,12
2,automation,6,5
4,generative AI,2,2
3,fairness,1,1


In [34]:
print_company_evidence(
    company_id="006",
    keywords=[
        "artificial intelligence",
        "machine learning",
        "generative AI",
        "large language model",
        "LLM",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness"
    ],
    max_items=35
)

Company: Aviva plc
Keyword: artificial intelligence
Page: 5
----------------------------------------------------------------------------------------------------
Insurance, Wealth and Driving operating leverage with Benefitting from our diversified capital-light and respect.Retirement solutions, technology and artificial intelligence (AI) portfolio, which drives resilient performance

Company: Aviva plc
Keyword: artificial intelligence
Page: 28
----------------------------------------------------------------------------------------------------
and Driving operating leverage and transforming with data and artificial intelligence.

Company: Aviva plc
Keyword: artificial intelligence
Page: 35
----------------------------------------------------------------------------------------------------
ingcorporates. Aviva access to underwrite risks in the and exploring new and innovative the use of artificial intelligence

Company: Aviva plc
Keyword: artificial intelligence
Page: 35
----------------

In [35]:
# Manual validation outcome for Aviva plc

aviva_mask = manual_review["company_id"] == "006"

manual_review.loc[
    aviva_mask,
    "manual_adjusted_score"
] = 10

manual_review.loc[
    aviva_mask,
    "manual_adjusted_percentage"
] = 71.43

manual_review.loc[
    aviva_mask,
    "reason_for_adjustment"
] = (
    "Score increased from provisional automated score. Aviva provides stronger AI-specific transparency evidence than the "
    "automated score captured, including references to artificial intelligence and generative AI in business transformation. The report also discusses "
    "AI-related risks and controls, cyber resilience, GenAI supplier risks, and the Group Risk Identification Process. Governance and risk scores were therefore increased. "
    "However, no clear evidence was found for human-in-the-loop review in AI-supported decisions. "
    "Fairness references appear mostly general rather than AI-specific, so the ethics/fairness score was not increased."
)

manual_review.loc[
    aviva_mask,
    "important_pages"
] = "5, 28, 34, 35, 40, 46, 80, 81, 84, 85, 101, 102, 109, 283, 338"

manual_review.loc[
    aviva_mask,
    "important_extracts"
] = (
    "Aviva refers to artificial intelligence as part of its technology portfolio and transformation with data and AI; "
    "the report mentions deploying 12 Generative AI solutions and launching a new Generative AI tool; "
    "the report discusses risks associated with adoption and reliance on rapidly advancing technologies such as AI and quantum change; "
    "it also references operational risk and control management, AI-related governance, the Group Risk Identification Process, "
    "supplier controls around GenAI risks, cyber resilience and Board or organisational review of AI-related skills and capability."
)

manual_review.loc[
    aviva_mask,
    "review_status"
] = "Reviewed - score increased"

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review[manual_review["company_id"] == "006"]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
9,006,Aviva plc,91,5,33,6,42.86,10.0,71.43,"Score increased from provisional automated score. Aviva provides stronger AI-specific transparency evidence than the automated score captured, including references to artificial intelligence and generative AI in business transformation. The report also discusses AI-related risks and controls, cyber resilience, GenAI supplier risks, and the Group Risk Identification Process. Governance and risk scores were therefore increased. However, no clear evidence was found for human-in-the-loop review in AI-supported decisions. Fairness references appear mostly general rather than AI-specific, so the ethics/fairness score was not increased.","5, 28, 34, 35, 40, 46, 80, 81, 84, 85, 101, 102, 109, 283, 338","Aviva refers to artificial intelligence as part of its technology portfolio and transformation with data and AI; the report mentions deploying 12 Generative AI solutions and launching a new Generative AI tool; the report discusses risks associated with adoption and reliance on rapidly advancing technologies such as AI and quantum change; it also references operational risk and control management, AI-related governance, the Group Risk Identification Process, supplier controls around GenAI risks, cyber resilience and Board or organisational review of AI-related skills and capability.",Reviewed - score increased


In [36]:
def show_contexts_for_company(company_id, keywords=None, max_items=35):
    """
    Displays extracted AI/transparency-related evidence for one company.
    Uses the manual_review dataframe.
    """

    company_id = str(company_id).zfill(3)

    df = manual_review.copy()
    df["company_id"] = df["company_id"].astype(str).str.zfill(3)

    company_df = df[df["company_id"] == company_id].copy()

    if keywords is not None:
        keywords_lower = [k.lower() for k in keywords]
        company_df = company_df[
            company_df["keyword"].astype(str).str.lower().isin(keywords_lower)
        ]

    company_df = company_df.sort_values(
        by=["keyword", "page_number"],
        ascending=[True, True]
    ).head(max_items)

    if company_df.empty:
        print(f"No matching contexts found for company_id {company_id}.")
        return

    for _, row in company_df.iterrows():
        print("=" * 100)
        print(f"Company: {row.get('company_name', '')}")
        print(f"Keyword: {row.get('keyword', '')}")
        print(f"Page: {row.get('page_number', '')}")
        print("-" * 100)
        print(row.get("context", ""))
        print()

In [37]:
# Reviewing AI transparency evidence for Legal & General Group plc

show_contexts_for_company(
    company_id="007",
    keywords=[
        "artificial intelligence",
        "AI",
        "machine learning",
        "generative AI",
        "GenAI",
        "algorithm",
        "automation",
        "automated decision-making",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "bias",
        "fairness",
        "model risk",
        "data science",
        "analytics"
    ],
    max_items=35
)

KeyError: 'keyword'

In [39]:
import pandas as pd
import os

In [40]:
manual_review.columns.tolist()

['company_id',
 'company_name',
 'total_ai_transparency_mentions',
 'unique_keywords',
 'unique_pages',
 'total_score',
 'score_percentage',
 'manual_adjusted_score',
 'manual_adjusted_percentage',
 'reason_for_adjustment',
 'important_pages',
 'important_extracts',
 'review_status']

In [41]:
ai_mentions.columns.tolist()

['company_id',
 'company_name',
 'report_year',
 'source_file',
 'category',
 'keyword',
 'search_variant',
 'page_number',
 'matched_text',
 'context']

In [42]:
def show_contexts_for_company(company_id, keywords=None, max_items=35):
    """
    Shows detailed AI transparency evidence for one company.
    Uses ai_mentions because this dataframe contains keyword, page_number, matched_text, and context.
    """

    df = ai_mentions.copy()

    df["company_id"] = df["company_id"].astype(str).str.zfill(3)
    company_id = str(company_id).zfill(3)

    company_df = df[df["company_id"] == company_id].copy()

    if keywords is not None:
        keywords_lower = [k.lower() for k in keywords]
        company_df = company_df[
            company_df["keyword"].astype(str).str.lower().isin(keywords_lower)
        ]

    company_df = company_df.sort_values(
        by=["keyword", "page_number"],
        ascending=[True, True]
    ).head(max_items)

    if company_df.empty:
        print(f"No matching evidence found for company_id {company_id}.")
        return

    for _, row in company_df.iterrows():
        print("=" * 100)
        print(f"Company: {row['company_name']}")
        print(f"Keyword: {row['keyword']}")
        print(f"Page: {row['page_number']}")
        print("-" * 100)
        print(row["context"])
        print()

In [43]:
show_contexts_for_company(
    company_id="007",
    keywords=[
        "artificial intelligence",
        "AI",
        "machine learning",
        "generative AI",
        "GenAI",
        "algorithm",
        "automation",
        "automated decision-making",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "bias",
        "fairness",
        "model risk",
        "data science",
        "analytics"
    ],
    max_items=35
)

Company: Legal & General Group plc
Keyword: AI
Page: 5
----------------------------------------------------------------------------------------------------
ust also respond to emerging trends, including developments in ArtificialIn February 2025, we announced a Intelligence (AI). In 2026, we plan to£1.8 billion transaction with Japanese introduce AI-powered features to someinsurer Meiji Yasuda Life, which saw them

Company: Legal & General Group plc
Keyword: AI
Page: 5
----------------------------------------------------------------------------------------------------
ficialIn February 2025, we announced a Intelligence (AI). In 2026, we plan to£1.8 billion transaction with Japanese introduce AI-powered features to someinsurer Meiji Yasuda Life, which saw them of our customer facing platforms withacquire our US protection business and, at

Company: Legal & General Group plc
Keyword: AI
Page: 5
----------------------------------------------------------------------------------------------

In [44]:
# Manual review for Legal & General Group plc

company_id = "007"

# Make sure company_id is treated consistently as a 3-digit string
manual_review["company_id"] = manual_review["company_id"].astype(str).str.zfill(3)

# Make sure text columns can accept written notes
text_cols = [
    "reason_for_adjustment",
    "important_pages",
    "important_extracts",
    "review_status"
]

for col in text_cols:
    if col in manual_review.columns:
        manual_review[col] = manual_review[col].astype("object")

mask = manual_review["company_id"] == company_id

# Keep the original score because the evidence supports the provisional ranking
manual_review.loc[mask, "manual_adjusted_score"] = manual_review.loc[mask, "total_score"]
manual_review.loc[mask, "manual_adjusted_percentage"] = manual_review.loc[mask, "score_percentage"]

manual_review.loc[mask, "reason_for_adjustment"] = (
    "No adjustment made. Legal & General shows strong AI transparency evidence, particularly around AI risk governance. "
    "The disclosures mention a dedicated AI Risk non-financial risk category, a Central AI Inventory, risk intake process, "
    "AI use-case triage by risk level, second-line AI risk oversight, and an AI governance programme. Some extracted contexts "
    "are repetitive, especially around page 48, but the underlying evidence supports retaining the provisional transparency score."
)

manual_review.loc[mask, "important_pages"] = "5, 7, 8, 24, 44, 48, 57, 70, 77"

manual_review.loc[mask, "important_extracts"] = (
    "Page 48: AI risk, privacy, operational disruption, dedicated AI Risk NFR category, Central AI Inventory, "
    "risk intake process, use-case triage, second-line AI risk oversight, and AI governance programme. "
    "Page 77: Data and AI strategy, cyber security, technology advisers, and technology implementation planning. "
    "Pages 5, 7, 8, 24 and 44: AI-powered features, customer outcomes, operational efficiency, productivity gains, "
    "and AI-enabled support."
)

manual_review.loc[mask, "review_status"] = "Reviewed - provisional score retained"

# Save updated manual review file
manual_review.to_csv(manual_review_file, index=False)

# Check the saved review row
manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
10,007,Legal & General Group plc,51,5,19,6,42.86,6.0,42.86,"No adjustment made. Legal & General shows strong AI transparency evidence, particularly around AI risk governance. The disclosures mention a dedicated AI Risk non-financial risk category, a Central AI Inventory, risk intake process, AI use-case triage by risk level, second-line AI risk oversight, and an AI governance programme. Some extracted contexts are repetitive, especially around page 48, but the underlying evidence supports retaining the provisional transparency score.","5, 7, 8, 24, 44, 48, 57, 70, 77","Page 48: AI risk, privacy, operational disruption, dedicated AI Risk NFR category, Central AI Inventory, risk intake process, use-case triage, second-line AI risk oversight, and AI governance programme. Page 77: Data and AI strategy, cyber security, technology advisers, and technology implementation planning. Pages 5, 7, 8, 24 and 44: AI-powered features, customer outcomes, operational efficiency, productivity gains, and AI-enabled support.",Reviewed - provisional score retained


In [45]:
# Moving to the next company for manual validation

# Making IDs consistent
manual_review["company_id"] = manual_review["company_id"].astype(str).str.zfill(3)
ai_mentions["company_id"] = ai_mentions["company_id"].astype(str).str.zfill(3)

# Identifying companies not yet manually reviewed
review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
11,014,Wise plc,29,6,10,6,42.86,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [46]:
# AI transparency evidence for the next unreviewed company

company_id = next_company["company_id"].iloc[0]
company_name = next_company["company_name"].iloc[0]

print("Next company for review:")
print(company_id, "-", company_name)

keywords = [
    "artificial intelligence",
    "AI",
    "machine learning",
    "generative AI",
    "GenAI",
    "algorithm",
    "automation",
    "automated decision-making",
    "responsible AI",
    "AI ethics",
    "ethical AI",
    "human oversight",
    "human review",
    "human-in-the-loop",
    "bias",
    "fairness",
    "model risk",
    "data science",
    "analytics"
]

keywords_lower = [k.lower() for k in keywords]

company_mentions = ai_mentions[
    (ai_mentions["company_id"] == company_id) &
    (ai_mentions["keyword"].astype(str).str.lower().isin(keywords_lower))
].copy()

company_mentions = company_mentions.sort_values(
    by=["keyword", "page_number"],
    ascending=[True, True]
).head(40)

for _, row in company_mentions.iterrows():
    print("=" * 100)
    print(f"Company: {row['company_name']}")
    print(f"Keyword: {row['keyword']}")
    print(f"Page: {row['page_number']}")
    print("-" * 100)
    print(row["context"])

Next company for review:
014 - Wise plc
Company: Wise plc
Keyword: AI
Page: 21
----------------------------------------------------------------------------------------------------
service, it should be as easy and frictionless as possible. We have invested in AI, as well as our servicing team, to reduce contact rates; last year bringing the contact rate per active user to around
Company: Wise plc
Keyword: AI
Page: 21
----------------------------------------------------------------------------------------------------
hours. We will continue to invest in improving the service we offer our customers and we are focusing on leveraging AI and automation tools to deliver a better experience and to keep customer money moving around the world
Company: Wise plc
Keyword: AI
Page: 23
----------------------------------------------------------------------------------------------------
the biggest areas of progress in What differentiates Servicing at Wise from How are you using automation and Servici

In [47]:
# Manual review for Wise plc

company_id = "014"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = manual_review.loc[mask, "total_score"]
manual_review.loc[mask, "manual_adjusted_percentage"] = manual_review.loc[mask, "score_percentage"]

manual_review.loc[mask, "reason_for_adjustment"] = (
    "No adjustment made. Wise shows repeated AI-related disclosures across operational use, "
    "automation, customer servicing, Large Language Models, model risk, AI governance, financial crime, "
    "privacy and vendor technology risk. Some extracts are fragmented, but the overall evidence supports "
    "retaining the provisional transparency score."
)

manual_review.loc[mask, "important_pages"] = "21, 23, 62, 67, 68, 69"

manual_review.loc[mask, "important_extracts"] = (
    "Wise discusses investment in AI and automation tools to improve customer service and reduce contact rates. "
    "The annual report refers to automation in servicing, use of Large Language Models and chat functions, "
    "processing high volumes of documents through automation and AI, and AI-supported customer support. "
    "Further evidence links AI to model risk, governance, risk analytics, AI Forum oversight, AI governance action plans, "
    "privacy, financial crime, phishing and vendor technology controls."
)

manual_review.loc[mask, "review_status"] = "Reviewed - provisional score retained"

manual_review.to_csv(manual_review_file, index=False)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
11,014,Wise plc,29,6,10,6,42.86,6.0,42.86,"No adjustment made. Wise shows repeated AI-related disclosures across operational use, automation, customer servicing, Large Language Models, model risk, AI governance, financial crime, privacy and vendor technology risk. Some extracts are fragmented, but the overall evidence supports retaining the provisional transparency score.","21, 23, 62, 67, 68, 69","Wise discusses investment in AI and automation tools to improve customer service and reduce contact rates. The annual report refers to automation in servicing, use of Large Language Models and chat functions, processing high volumes of documents through automation and AI, and AI-supported customer support. Further evidence links AI to model risk, governance, risk analytics, AI Forum oversight, AI governance action plans, privacy, financial crime, phishing and vendor technology controls.",Reviewed - provisional score retained


In [48]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
12,015,CAB Payments Holdings plc,28,6,16,6,42.86,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [49]:
# Reviewing AI transparency evidence for CAB Payments Holdings plc

show_contexts_for_company(
    company_id="015",
    keywords=[
        "artificial intelligence",
        "AI",
        "machine learning",
        "generative AI",
        "GenAI",
        "algorithm",
        "automation",
        "automated decision-making",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "bias",
        "fairness",
        "model risk",
        "data science",
        "analytics"
    ],
    max_items=35
)

Company: CAB Payments Holdings plc
Keyword: AI
Page: 11
----------------------------------------------------------------------------------------------------
ategic Report Governance Statements Appendix 9 Chair’s Statement Embarking upon sustainable growth following a year of reset including AI, and diversified our revenue making as we support the executive team in streams. We are now more efficient, resilient delivering our purpose and strategy.

Company: CAB Payments Holdings plc
Keyword: AI
Page: 16
----------------------------------------------------------------------------------------------------
Renminbi (CNY) to offer more non- as fraud detection, compliance, and ■Invested in AI and automation to USD flexibility to clients customer service, ultimately reducing opti

Company: CAB Payments Holdings plc
Keyword: AI
Page: 16
----------------------------------------------------------------------------------------------------
ary 2026 partnerships, digital currency solutions, and AI-dr

In [54]:
# Manual review for CAB Payments Holdings plc

company_id = "015"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = manual_review.loc[mask, "total_score"]
manual_review.loc[mask, "manual_adjusted_percentage"] = manual_review.loc[mask, "score_percentage"]

manual_review.loc[mask, "reason_for_adjustment"] = (
    "No adjustment made. CAB Payments shows moderate AI transparency evidence, including AI and automation investment, "
    "machine learning capabilities, alert handling, sanctions screening, fraud detection, compliance, customer service, "
    "and references to an AI Policy and Board-level responsibility. Some fairness and bias matches are noisy or unrelated "
    "to AI governance, but the overall evidence supports retaining the provisional transparency score."
)

manual_review.loc[mask, "important_pages"] = "11, 16, 25, 27, 37, 48, 50, 73, 74"

manual_review.loc[mask, "important_extracts"] = (
    "CAB refers to investing in AI and automation for fraud detection, compliance and customer service. "
    "The report mentions machine learning capabilities and improved alert handling. "
    "Evidence also refers to responsible adoption of AI, an AI Policy, and Board responsibility for AI governance. "
    "Further extracts link AI to technology infrastructure, cyber security enhancements and operational efficiency."
)

manual_review.loc[mask, "review_status"] = "Reviewed - provisional score retained"

manual_review.to_csv(manual_review_file, index=False)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
12,015,CAB Payments Holdings plc,28,6,16,6,42.86,6.0,42.86,"No adjustment made. CAB Payments shows moderate AI transparency evidence, including AI and automation investment, machine learning capabilities, alert handling, sanctions screening, fraud detection, compliance, customer service, and references to an AI Policy and Board-level responsibility. Some fairness and bias matches are noisy or unrelated to AI governance, but the overall evidence supports retaining the provisional transparency score.","11, 16, 25, 27, 37, 48, 50, 73, 74","CAB refers to investing in AI and automation for fraud detection, compliance and customer service. The report mentions machine learning capabilities and improved alert handling. Evidence also refers to responsible adoption of AI, an AI Policy, and Board responsibility for AI governance. Further extracts link AI to technology infrastructure, cyber security enhancements and operational efficiency.",Reviewed - provisional score retained


In [51]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
13,008,M&G plc,24,5,19,6,42.86,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [52]:
# Reviewing AI transparency evidence for M&G plc

show_contexts_for_company(
    company_id="008",
    keywords=[
        "artificial intelligence",
        "AI",
        "machine learning",
        "generative AI",
        "GenAI",
        "automation",
        "responsible AI",
        "AI ethics",
        "ethical AI",
        "human oversight",
        "human review",
        "human-in-the-loop",
        "automated decision-making",
        "bias",
        "fairness",
        "model risk",
        "data science",
        "analytics"
    ],
    max_items=35
)

Company: M&G plc
Keyword: AI
Page: 8
----------------------------------------------------------------------------------------------------
with model, international the use of advanced technology. AI will be an enabler for growth and we remain focused on footprint and depth of our increasing adoption across the business to strengthen investment expertise w

Company: M&G plc
Keyword: AI
Page: 15
----------------------------------------------------------------------------------------------------
5-2027. – Simplify and automate our processes, using technology – Leverage the strength of our business model to develop – Maintain a progressive and sustainable dividend policy. and AI, to improve efficiency, service and customer innovative products and investment solutions to meet experience.

Company: M&G plc
Keyword: AI
Page: 19
----------------------------------------------------------------------------------------------------
l Our efforts have delivered significant improvements – Advice: At

In [57]:
# Manual review for M&G plc

company_id = "008"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = manual_review.loc[mask, "total_score"]
manual_review.loc[mask, "manual_adjusted_percentage"] = manual_review.loc[mask, "score_percentage"]

manual_review.loc[mask, "reason_for_adjustment"] = (
    "No adjustment made. M&G shows moderate AI transparency evidence through references to AI in strategy, The company also refers to adopting AI solutions in a considered manner, "
    "supported by an AI framework, governance structures, oversight and training. Some matches, are noisy or only indirectly relevant, but the stronger governance and usage "
    "evidence supports retaining the provisional transparency score."
)

manual_review.loc[mask, "important_pages"] = "8, 15, 19, 38, 43, 48, 91, 93, 95"

manual_review.loc[mask, "important_extracts"] = (
    "M&G describes AI as an enabler for growth and links AI adoption to strengthening investment expertise. "
    "The report refers to simplifying and automating processes using technology and AI to improve efficiency, service and customer experience. "
    "It also mentions AI-powered support and technical resources for routine tasks, Board-level review of AI strategy, "
    "and third-party supplier risks linked to artificial intelligence. "
)

manual_review.loc[mask, "review_status"] = "Reviewed - provisional score retained"

manual_review.to_csv(manual_review_file, index=False)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
13,008,M&G plc,24,5,19,6,42.86,6.0,42.86,"No adjustment made. M&G shows moderate AI transparency evidence through references to AI in strategy, The company also refers to adopting AI solutions in a considered manner, supported by an AI framework, governance structures, oversight and training. Some matches, are noisy or only indirectly relevant, but the stronger governance and usage evidence supports retaining the provisional transparency score.","8, 15, 19, 38, 43, 48, 91, 93, 95","M&G describes AI as an enabler for growth and links AI adoption to strengthening investment expertise. The report refers to simplifying and automating processes using technology and AI to improve efficiency, service and customer experience. It also mentions AI-powered support and technical resources for routine tasks, Board-level review of AI strategy, and third-party supplier risks linked to artificial intelligence.",Reviewed - provisional score retained


In [58]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
14,011,IG Group Holdings plc,22,5,16,6,42.86,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [59]:
# AI transparency evidence for the next company

company_id = next_company["company_id"].astype(str).str.zfill(3).iloc[0]
company_name = next_company["company_name"].iloc[0]

print(f"Next company for review:")
print(f"{company_id} - {company_name}")
print("=" * 100)

keywords = [
    "artificial intelligence",
    "AI",
    "machine learning",
    "generative AI",
    "GenAI",
    "algorithm",
    "automation",
    "automated decision-making",
    "responsible AI",
    "AI ethics",
    "ethical AI",
    "human oversight",
    "human review",
    "human-in-the-loop",
    "bias",
    "fairness",
    "model risk",
    "data science",
    "analytics"
]

keywords_lower = [k.lower() for k in keywords]

company_extracts = ai_mentions[
    (ai_mentions["company_id"].astype(str).str.zfill(3) == company_id) &
    (ai_mentions["keyword"].astype(str).str.lower().isin(keywords_lower))
].sort_values(
    by=["keyword", "page_number"],
    ascending=[True, True]
).head(35)

for _, row in company_extracts.iterrows():
    print("=" * 100)
    print(f"Company: {row['company_name']}")
    print(f"Keyword: {row['keyword']}")
    print(f"Page: {row['page_number']}")
    print("-" * 100)
    print(row["context"])

Next company for review:
011 - IG Group Holdings plc
Company: IG Group Holdings plc
Keyword: AI
Page: 20
----------------------------------------------------------------------------------------------------
FY23 Reported 358.5 FY23 Reported 72.6 FY23 35% In FY25 we invested in an agile, AI-powered employee The number of active customers is the most relevant metric for First trades refers to a new customer funding their account and Our gender diversity metric represents the percentage
Company: IG Group Holdings plc
Keyword: AI
Page: 47
----------------------------------------------------------------------------------------------------
igate the ever-present and changing processes, people, systems, cyber threats or external events.  AI controls ensure that any sensitive information is only used in private, paid for and licensed services so as not to be mad
Company: IG Group Holdings plc
Keyword: AI
Page: 47
---------------------------------------------------------------------------------

In [61]:
# Manual review for IG Group Holdings plc

company_id = "011"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = manual_review.loc[mask, "total_score"]
manual_review.loc[mask, "manual_adjusted_percentage"] = manual_review.loc[mask, "score_percentage"]

manual_review.loc[mask, "reason_for_adjustment"] = (
    "No adjustment made. IG Group shows relevant AI transparency evidence, including AI-powered employee "
    "support, automation and artificial intelligence across growing markets, AI controls for sensitive "
    "information, AI-related operational and customer service opportunities, risk management oversight, "
    "and Board attention to AI, cybersecurity and crypto-related risks. Some fairness and bias "
    "matches are less directly connected to AI governance, but the overall evidence supports retaining "
    "the provisional transparency score."
)

manual_review.loc[mask, "important_pages"] = "11, 12, 17, 20, 47, 48, 61, 72, 73, 86, 87, 88"

manual_review.loc[mask, "important_extracts"] = (
    "IG refers to investment in an agile, AI-powered employee metric. "
    "The report states that AI controls help ensure sensitive information is used appropriately. "
    "It discusses automation and artificial intelligence across growing markets, including digital servicing and customer support. "
    "Board and committee materials refer to AI, cybersecurity, crypto, risk appetite, and ongoing reporting on AI opportunities and risks."
)

manual_review.loc[mask, "review_status"] = "Reviewed - provisional score retained"

manual_review.to_csv(manual_review_file, index=False)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
14,011,IG Group Holdings plc,22,5,16,6,42.86,6.0,42.86,"No adjustment made. IG Group shows relevant AI transparency evidence, including AI-powered employee support, automation and artificial intelligence across growing markets, AI controls for sensitive information, AI-related operational and customer service opportunities, risk management oversight, and Board attention to AI, cybersecurity and crypto-related risks. Some fairness and bias matches are less directly connected to AI governance, but the overall evidence supports retaining the provisional transparency score.","11, 12, 17, 20, 47, 48, 61, 72, 73, 86, 87, 88","IG refers to investment in an agile, AI-powered employee metric. The report states that AI controls help ensure sensitive information is used appropriately. It discusses automation and artificial intelligence across growing markets, including digital servicing and customer support. Board and committee materials refer to AI, cybersecurity, crypto, risk appetite, and ongoing reporting on AI opportunities and risks.",Reviewed - provisional score retained


In [62]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
15,010,AJ Bell plc,21,6,14,6,42.86,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [66]:
# Reviewing AI transparency evidence for AJ Bell plc

company_id = "010"

keywords = [
    "artificial intelligence",
    "AI",
    "machine learning",
    "generative AI",
    "GenAI",
    "large language model",
    "LLM",
    "algorithm",
    "automation",
    "automated decision-making",
    "responsible AI",
    "AI ethics",
    "ethical AI",
    "human oversight",
    "human review",
    "human-in-the-loop",
    "bias",
    "fairness",
    "model risk",
    "data science",
    "analytics"
]

company_extracts = ai_mentions[
    ai_mentions["company_id"].astype(str).str.zfill(3) == company_id
].copy()

company_extracts = company_extracts[
    company_extracts["keyword"].astype(str).str.lower().isin(
        [keyword.lower() for keyword in keywords]
    )
]

company_extracts = company_extracts.sort_values(
    by=["page_number", "keyword"],
    ascending=[True, True]
)

print("Company: AJ Bell plc")
print("Company ID:", company_id)
print("Extracts found:", len(company_extracts))
print("=" * 100)

for _, row in company_extracts.iterrows():
    print(f"Company: {row['company_name']}")
    print(f"Keyword: {row['keyword']}")
    print(f"Page: {row['page_number']}")
    print("-" * 100)
    print(row["context"])
    print("=" * 100)

Company: AJ Bell plc
Company ID: 010
Extracts found: 21
Company: AJ Bell plc
Keyword: AI
Page: 7
----------------------------------------------------------------------------------------------------
service, allowing our customers to invest when they choose. Increasingly, we leverage generative AI (GenAI) to streamline back-office processes, enabling us to
Company: AJ Bell plc
Keyword: generative AI
Page: 7
----------------------------------------------------------------------------------------------------
service, allowing our customers to invest when they choose. Increasingly, we leverage generative AI (GenAI) to streamline back-office processes, enabling us to
Company: AJ Bell plc
Keyword: automation
Page: 12
----------------------------------------------------------------------------------------------------
in proposition, in June 2025 and are enhancing cash further growth in FY26 and beyond. the pension tax system throughout this Parliament, management and investment automation on 

In [64]:
# Manual review for AJ Bell plc

company_id = "010"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = manual_review.loc[
    mask, "total_score"
]

manual_review.loc[mask, "manual_adjusted_percentage"] = manual_review.loc[
    mask, "score_percentage"
]

manual_review.loc[mask, "reason_for_adjustment"] = (
    "No adjustment made. AJ Bell provides moderate and relevant AI transparency "
    "evidence, The report also indicates review of investment in automation. However, the disclosures provide limited "
    "detail on AI-specific governance. Several fairness and bias matches relate to employment, "
    "remuneration and workplace inclusion rather than AI, while the page 164 "
    "matches are glossary references. The overall evidence therefore supports "
    "retaining the provisional transparency score."
)

manual_review.loc[mask, "important_pages"] = "7, 19, 41, 95"

manual_review.loc[mask, "important_extracts"] = (
    "The report describes the integration of GenAI across key business processes, "
    "robotics in back-office operations and an AI-supported single-customer-view "
    "dashboard used by the Customer Service Team. "
    "Committee materials indicate that investment in automation was reviewed as "
    "part of governance and risk discussions."
)

manual_review.loc[mask, "review_status"] = (
    "Reviewed - provisional score retained"
)

manual_review.to_csv(manual_review_file, index=False)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
15,010,AJ Bell plc,21,6,14,6,42.86,6.0,42.86,"No adjustment made. AJ Bell provides moderate and relevant AI transparency evidence, The report also indicates review of investment in automation. However, the disclosures provide limited detail on AI-specific governance. Several fairness and bias matches relate to employment, remuneration and workplace inclusion rather than AI, while the page 164 matches are glossary references. The overall evidence therefore supports retaining the provisional transparency score.","7, 19, 41, 95","The report describes the integration of GenAI across key business processes, robotics in back-office operations and an AI-supported single-customer-view dashboard used by the Customer Service Team. Committee materials indicate that investment in automation was reviewed as part of governance and risk discussions.",Reviewed - provisional score retained


In [65]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
16,017,Boku Inc.,23,3,10,5,35.71,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [67]:
# Reviewing AI transparency evidence for Boku Inc.

company_id = "017"

company_extracts = ai_mentions[
    ai_mentions["company_id"].astype(str).str.zfill(3) == company_id
].copy()

company_extracts = company_extracts.sort_values(
    by=["keyword", "page_number"],
    ascending=[True, True]
)

print("Company: Boku Inc.")
print("Company ID:", company_id)
print("Extracts found:", len(company_extracts))
print("=" * 90)

for _, row in company_extracts.iterrows():
    print(f"Company: {row['company_name']}")
    print(f"Keyword: {row['keyword']}")
    print(f"Page: {row['page_number']}")
    print("-" * 90)
    print(row["context"])
    print("=" * 90)

Company: Boku Inc.
Company ID: 017
Extracts found: 23
Company: Boku Inc.
Keyword: AI
Page: 4
------------------------------------------------------------------------------------------
utomation and scalability through +30% vs 2024 (+29% CER) operational infrastructure enhancements Growing EBITDA $41.3m +36% vs 2024 Invested in key capabilities including data, AI, product and transformation to drive scalability Operating proﬁt $18.9M +205% vs 2024 Launched Singapore Innovation Hub B
Company: Boku Inc.
Keyword: AI
Page: 10
------------------------------------------------------------------------------------------
clear priority. • Innovate: Innovation remains central to our culture and a core differentiator for Boku. During the year, we advanced our platform beyond core payment processing, investing in automation, AI and data-driven capabilities to enhance transaction conversion rates, settlement speed and scalability for our merchants, while reducing friction across the payment journey. 

In [68]:
# Manual review for Boku Inc.

company_id = "017"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = 7.0
manual_review.loc[mask, "manual_adjusted_percentage"] = 50.00

manual_review.loc[mask, "reason_for_adjustment"] = (
    "Score adjusted upward from 5 to 7. Boku provides meaningful AI transparency evidence, "
    "including the intentional and responsible incorporation of AI tools. The report also refers to "
    "AI-related regulatory developments, compliance monitoring and operational resilience. "
    "However, disclosure remains limited regarding human oversight, model testing, explainability, "
    "bias controls, performance monitoring and formal accountability. Fairness references relate "
    "mainly to remuneration and workforce matters."
)

manual_review.loc[mask, "important_pages"] = "4, 10, 12, 19, 20, 21, 23"

manual_review.loc[mask, "important_extracts"] = (
    "Boku reports investment in data, AI and transformation capabilities to improve scalability. "
    "It describes the intentional and responsible incorporation of AI tools, including agentic AI, "
    "across fraud and risk management, customer support, operational automation, data analysis and "
    "product development. AI is presented as an enabler of better decision-making, faster execution "
    "and improved outcomes. Further extracts refer to AI-enabled settlement, operational resilience, "
    "emerging AI regulation and strengthened compliance monitoring."
)

manual_review.loc[mask, "review_status"] = "Reviewed - score manually adjusted"

manual_review.to_csv(manual_review_file, index=False)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
16,017,Boku Inc.,23,3,10,5,35.71,7.0,50.0,"Score adjusted upward from 5 to 7. Boku provides meaningful AI transparency evidence, including the intentional and responsible incorporation of AI tools. The report also refers to AI-related regulatory developments, compliance monitoring and operational resilience. However, disclosure remains limited regarding human oversight, model testing, explainability, bias controls, performance monitoring and formal accountability. Fairness references relate mainly to remuneration and workforce matters.","4, 10, 12, 19, 20, 21, 23","Boku reports investment in data, AI and transformation capabilities to improve scalability. It describes the intentional and responsible incorporation of AI tools, including agentic AI, across fraud and risk management, customer support, operational automation, data analysis and product development. AI is presented as an enabler of better decision-making, faster execution and improved outcomes. Further extracts refer to AI-enabled settlement, operational resilience, emerging AI regulation and strengthened compliance monitoring.",Reviewed - score manually adjusted


In [69]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
17,013,IntegraFin Holdings plc,11,3,3,3,21.43,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [70]:
# Reviewing AI transparency evidence for IntegraFin Holdings plc

company_id = "013"

company_extracts = ai_mentions[
    ai_mentions["company_id"].astype(str).str.zfill(3) == company_id
].copy()

company_extracts = company_extracts.sort_values(
    by=["page_number", "keyword"],
    ascending=[True, True]
)

print("Company:", company_extracts["company_name"].iloc[0])
print("Company ID:", company_id)
print("Extracts found:", len(company_extracts))
print("=" * 90)

for _, row in company_extracts.iterrows():
    print(f"Company: {row['company_name']}")
    print(f"Keyword: {row['keyword']}")
    print(f"Page: {row['page_number']}")
    print("-" * 90)
    print(row["context"])
    print("=" * 90)

Company: IntegraFin Holdings plc
Company ID: 013
Extracts found: 11
Company: IntegraFin Holdings plc
Keyword: AI
Page: 10
------------------------------------------------------------------------------------------
heir the adviser platform market to grow at 12% propositions and our overall IHP integrations data and unique access to AI capabilities. per annum from £806 billion at the end of strategy are aligned to these trends.
Company: IntegraFin Holdings plc
Keyword: AI
Page: 53
------------------------------------------------------------------------------------------
umer behaviour and confidence as well as increasing Time horizon: ongoing the risk of civil unrest and cyber attacks, particularly state sponsored or politically motivated cyber attacks. Artificial intelligence AI is a constantly evolving opportunity and risk for the Group. Key areas of activity have included business Time horizon: ongoing proposition and competitive advantage, day-to-day operations, workforce education, 

In [71]:
# Manual review for IntegraFin Holdings plc

company_id = "013"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = 5.0
manual_review.loc[mask, "manual_adjusted_percentage"] = round(
    (5 / 14) * 100,
    2
)

manual_review.loc[mask, "reason_for_adjustment"] = (
    "Manual upward adjustment made. IntegraFin's provisional score understates the "
    "substantive AI transparency evidence contained mainly on page 53. The report states "
    "that AI adoption occurs within a governance framework, AI use is continuously "
    "monitored, AI-awareness training is provided, and societal, ethical and industry "
    "impacts are actively monitored. It also identifies data governance as an important "
    "support for positive AI outcomes. Most extracts are overlapping passages "
    "from the same page, and the report does not clearly disclose human review, "
    "explainability, model testing, bias controls or formal AI audit arrangements. "
    "The score is therefore increased conservatively from 3 to 5."
)

manual_review.loc[mask, "important_pages"] = "10, 53"

manual_review.loc[mask, "important_extracts"] = (
    "IntegraFin identifies AI as an evolving opportunity and risk affecting business "
    "operations, competitive advantage, workforce education, cybersecurity, data "
    "protection and ethical or social considerations. "
    "The report states that employees should be AI-informed and that AI-awareness "
    "training forms a core part of adoption. "
    "AI adoption is described as occurring within a governance framework, supported by "
    "continuous monitoring of AI use, active monitoring of societal and ethical impacts, "
    "and data-governance practices."
)

manual_review.loc[mask, "review_status"] = (
    "Reviewed - manual score increased from 3 to 5"
)

manual_review.to_csv(
    manual_review_file,
    index=False
)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
17,013,IntegraFin Holdings plc,11,3,3,3,21.43,5.0,35.71,"Manual upward adjustment made. IntegraFin's provisional score understates the substantive AI transparency evidence contained mainly on page 53. The report states that AI adoption occurs within a governance framework, AI use is continuously monitored, AI-awareness training is provided, and societal, ethical and industry impacts are actively monitored. It also identifies data governance as an important support for positive AI outcomes. Most extracts are overlapping passages from the same page, and the report does not clearly disclose human review, explainability, model testing, bias controls or formal AI audit arrangements. The score is therefore increased conservatively from 3 to 5.","10, 53","IntegraFin identifies AI as an evolving opportunity and risk affecting business operations, competitive advantage, workforce education, cybersecurity, data protection and ethical or social considerations. The report states that employees should be AI-informed and that AI-awareness training forms a core part of adoption. AI adoption is described as occurring within a governance framework, supported by continuous monitoring of AI use, active monitoring of societal and ethical impacts, and data-governance practices.",Reviewed - manual score increased from 3 to 5


In [72]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
18,018,PayPoint plc,13,3,9,2,14.29,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [73]:
# Reviewing AI transparency evidence for PayPoint plc

company_id = "018"
company_name = "PayPoint plc"

paypoint_extracts = ai_mentions[
    ai_mentions["company_id"].astype(str).str.zfill(3) == company_id
].copy()

paypoint_extracts = paypoint_extracts.sort_values(
    by=["page_number", "keyword"],
    ascending=[True, True]
)

print(f"Company: {company_name}")
print(f"Company ID: {company_id}")
print(f"Extracts found: {len(paypoint_extracts)}")

for _, row in paypoint_extracts.iterrows():
    print("=" * 100)
    print(f"Company: {row['company_name']}")
    print(f"Keyword: {row['keyword']}")
    print(f"Page: {row['page_number']}")
    print("-" * 100)
    print(row["context"])

Company: PayPoint plc
Company ID: 018
Extracts found: 13
Company: PayPoint plc
Keyword: automation
Page: 5
----------------------------------------------------------------------------------------------------
organisational a reduction of at least in the range of 5% to framework which 20% of our issued share 8% per annum across will deliver greater capital, with scope for the Group automation of processes leverage in the range and greater agility to of 1.2x to 1.5x support the delivery of our plan Our strategy over the past five ye
Company: PayPoint plc
Keyword: automation
Page: 5
----------------------------------------------------------------------------------------------------
in the range of 1.2x to 1.5x. growth rate in the range of 5-8% per annum. operational structure and business processes today and develop a plan to deliver greater automation and business agility in the future to support the delivery of our plan.
Company: PayPoint plc
Keyword: automation
Page: 9
----------------

In [74]:
# Manual review for PayPoint plc

company_id = "018"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = manual_review.loc[
    mask, "total_score"
]

manual_review.loc[mask, "manual_adjusted_percentage"] = manual_review.loc[
    mask, "score_percentage"
]

manual_review.loc[mask, "reason_for_adjustment"] = (
    "No adjustment made. PayPoint shows limited but relevant AI transparency evidence, "
    "including the operational use of an AI-driven statement reader and acknowledgement "
    "of risks associated with the evolution of AI. Most of the remaining extracts "
    "refer to general process automation and organisational efficiency rather than AI-specific "
    "governance. The report does not clearly disclose human oversight, explainability, model "
    "testing, AI fairness controls or formal accountability arrangements. The bias match relates "
    "to accounting estimates rather than algorithmic bias. The available evidence therefore "
    "supports retaining the provisional transparency score."
)

manual_review.loc[mask, "important_pages"] = "14, 49, 56, 61"

manual_review.loc[mask, "important_extracts"] = (
    "PayPoint reports the use of an AI-driven statement reader within its retailer and payments services. "
    "The statement reader is presented as an operational tool intended to improve service delivery and efficiency. "
    "The report also recognises risks associated with the evolution of AI and refers to monitoring technological "
    "developments and using partnerships to help mitigate these risks."
)

manual_review.loc[mask, "review_status"] = "Reviewed - provisional score retained"

manual_review.to_csv(manual_review_file, index=False)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
18,018,PayPoint plc,13,3,9,2,14.29,2.0,14.29,"No adjustment made. PayPoint shows limited but relevant AI transparency evidence, including the operational use of an AI-driven statement reader and acknowledgement of risks associated with the evolution of AI. Most of the remaining extracts refer to general process automation and organisational efficiency rather than AI-specific governance. The report does not clearly disclose human oversight, explainability, model testing, AI fairness controls or formal accountability arrangements. The bias match relates to accounting estimates rather than algorithmic bias. The available evidence therefore supports retaining the provisional transparency score.","14, 49, 56, 61",PayPoint reports the use of an AI-driven statement reader within its retailer and payments services. The statement reader is presented as an operational tool intended to improve service delivery and efficiency. The report also recognises risks associated with the evolution of AI and refers to monitoring technological developments and using partnerships to help mitigate these risks.,Reviewed - provisional score retained


In [75]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
19,012,CMC Markets plc,4,4,4,1,7.14,NaN,NaN,NaN,NaN,NaN,Pending manual review


In [76]:
# Displaying all extracts for CMC Markets plc

company_id = "012"

company_extracts = ai_mentions[
    ai_mentions["company_id"].astype(str).str.zfill(3) == company_id
].copy()

company_name = (
    company_extracts["company_name"].iloc[0]
    if not company_extracts.empty
    else "CMC Markets plc"
)

print(f"Company: {company_name}")
print(f"Company ID: {company_id}")
print(f"Extracts found: {len(company_extracts)}")
print("=" * 100)

for _, row in company_extracts.iterrows():
    print(f"Company: {row['company_name']}")
    print(f"Keyword: {row['keyword']}")
    print(f"Page: {row['page_number']}")
    print("-" * 100)
    print(row["context"])
    print("=" * 100)

Company: CMC Markets plc
Company ID: 012
Extracts found: 4
Company: CMC Markets plc
Keyword: artificial intelligence
Page: 25
----------------------------------------------------------------------------------------------------
os. Operational risks | Risks arising from our people, processes, systems and external service providers Emerging risks We monitor emerging regulatory developments and technological advancements, including the rise of artificial intelligence and broader digital disruption. These trends have the potential to reshape how financial services are delivered and consumed. As part of our strategy, we aim to adapt our platforms, processes and product offerings to re
Company: CMC Markets plc
Keyword: AI
Page: 32
----------------------------------------------------------------------------------------------------
lity principles ypilla Our ESG strategy rs aligns with several United Ourthree sust ai nabilit Nations Sustainable Development Goals The Board oversees the conduct 

In [78]:
# Manual review for CMC Markets plc

company_id = "012"

mask = manual_review["company_id"].astype(str).str.zfill(3) == company_id

manual_review.loc[mask, "manual_adjusted_score"] = manual_review.loc[
    mask, "total_score"
]

manual_review.loc[mask, "manual_adjusted_percentage"] = manual_review.loc[
    mask, "score_percentage"
]

manual_review.loc[mask, "reason_for_adjustment"] = (
    "No adjustment made. CMC Markets provides limited AI transparency evidence. "
    "The report identifies artificial intelligence and wider digital disruption as emerging "
    "operational risks and indicates that the Group intends to adapt its platforms, processes "
    "and product offerings. It also reports an operational automation and robotics rollout. "
    "The disclosure remains general and does not provide detailed evidence of an AI "
    "The page 32 AI match appears to be an OCR or keyword false positive. The limited valid evidence therefore "
    "supports retaining the provisional transparency score."
)

manual_review.loc[mask, "important_pages"] = "25, 74"

manual_review.loc[mask, "important_extracts"] = (
    "Page 25 identifies artificial intelligence and broader digital disruption as emerging "
    "operational risks with the potential to reshape financial services. "
    "Page 74 refers to the rollout of automation and robotics across business operations. "
    "The remaining AI and bias matches are not directly relevant to substantive AI transparency."
)

manual_review.loc[mask, "review_status"] = (
    "Reviewed - provisional score retained"
)

manual_review.to_csv(manual_review_file, index=False)

manual_review.loc[mask]

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status
19,012,CMC Markets plc,4,4,4,1,7.14,1.0,7.14,"No adjustment made. CMC Markets provides limited AI transparency evidence. The report identifies artificial intelligence and wider digital disruption as emerging operational risks and indicates that the Group intends to adapt its platforms, processes and product offerings. It also reports an operational automation and robotics rollout. The disclosure remains general and does not provide detailed evidence of an AI The page 32 AI match appears to be an OCR or keyword false positive. The limited valid evidence therefore supports retaining the provisional transparency score.","25, 74",Page 25 identifies artificial intelligence and broader digital disruption as emerging operational risks with the potential to reshape financial services. Page 74 refers to the rollout of automation and robotics across business operations. The remaining AI and bias matches are not directly relevant to substantive AI transparency.,Reviewed - provisional score retained


In [79]:
# Moving to the next company for manual validation

review_status = manual_review["review_status"].fillna("").astype(str)

next_company = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
].head(1)

next_company

,company_id,company_name,total_ai_transparency_mentions,unique_keywords,unique_pages,total_score,score_percentage,manual_adjusted_score,manual_adjusted_percentage,reason_for_adjustment,important_pages,important_extracts,review_status


In [80]:
# Final check: confirming that all companies have been manually reviewed

review_status = manual_review["review_status"].fillna("").astype(str)

pending_reviews = manual_review[
    ~review_status.str.contains("Reviewed", case=False, na=False)
]

reviewed_count = len(manual_review) - len(pending_reviews)
total_count = len(manual_review)

print(f"Companies reviewed: {reviewed_count} out of {total_count}")

if pending_reviews.empty:
    print("Manual validation complete — all companies have been reviewed.")
else:
    print("The following companies still require review:")
    display(
        pending_reviews[
            ["company_id", "company_name", "review_status"]
        ]
    )

Companies reviewed: 20 out of 20
Manual validation complete — all companies have been reviewed.


In [81]:
# Saving the completed manual-review table

manual_review.to_csv(manual_review_file, index=False)

print(f"Completed manual review saved to: {manual_review_file}")
print(f"Rows saved: {len(manual_review)}")

Completed manual review saved to: ..\data\manual_review_notes.csv
Rows saved: 20
